# 第15章　分布外検出と安全な棄却 ― 「診るべきでない画像」を見分ける**『医療診断支援AIの社会実装（社会実装編）』のコード**本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**リポジトリ: https://github.com/kewel-corp/book-social

## OODスコアを、実装で捉える

In [ ]:
# マハラノビス距離によるOODスコア（学習後に一度だけ統計を推定）feats, labels = extract_penultimate(model, train_loader)   # 学習データの特徴とラベル# 平均は「クラスごと」に取る。全データで一つだけ推定すると、クラスが分かれている# 特徴空間では“クラスとクラスの谷間”が中心になり、そこに落ちたOODを見逃す。mus = torch.stack([feats[labels == c].mean(0) for c in range(num_classes)])centered = torch.cat([feats[labels == c] - mus[c] for c in range(num_classes)])cov_inv = torch.linalg.pinv(torch.cov(centered.T))  # 共分散はクラス共通（tied）で一つdef ood_score(x):    z = extract_penultimate(model, x)              # 入力の特徴    d = z[:, None, :] - mus[None, :, :]            # (B, C, D)    m = torch.einsum('bcd,de,bce->bc', d, cov_inv, d)    return m.min(dim=1).values                     # 最も近いクラスまでの距離＝OODスコア

## 誤り率を「保証」する ― コンフォーマル予測

In [ ]:
scores = 1 - softmax(model(cal_x))[range(n), cal_y]      # 較正データの非適合スコア# 較正例が少ないと分位点が1を超えて例外になる。n >= ceil(1/alpha) - 1 が必要q = np.quantile(scores, np.ceil((n+1)*(1-alpha))/n)      # 例: alpha=0.1 で90%保証def predict_set(x):    p = softmax(model(x))    return np.where(1 - p <= q, 1, 0)                     # 集合に含めるクラス

## 不確実性を二つに分ける ― aleatoric と epistemic

In [ ]:
# model.train() を呼んではいけない。BatchNormまで学習モードになり、# 推論のたびに running統計が書き換わって、以後の通常推論の出力まで変質する。# 効かせたいのはドロップアウトだけなので、実装編の不確実性と棄却の章の enable_dropout を使う。model.eval()enable_dropout(model)                             # Dropout モジュールだけ train に戻すwith torch.no_grad():    probs = torch.stack([softmax(model(x)) for _ in range(T)])  # (T, B, C)mean_p = probs.mean(0)total = entropy(mean_p)                           # 総不確実性aleatoric = entropy(probs, dim=-1).mean(0)        # 各予測のエントロピー平均epistemic = total - aleatoric                     # 差分＝モデルの無知model.eval()                                      # 使い終わったら必ず戻す